# 02 — ABC / Pareto Product Analysis

Classic inventory-management technique applied to the product catalogue: rank products by revenue contribution and split them into three tiers.

- **A products** — the vital few, roughly the top ~80% of cumulative revenue. Highest priority for stock availability, demand planning, and supplier negotiation.
- **B products** — the next ~15% of cumulative revenue. Moderate priority.
- **C products** — the remaining ~5% of cumulative revenue, but typically the majority of SKUs by count. Candidates for simplified/less frequent ordering, or portfolio rationalization.

This turns a descriptive statement like *"Product X had the highest sales"* into an actionable inventory-management insight: *"the top N% of SKUs drive X% of revenue — here's how to prioritize replenishment and review effort accordingly."*

---


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 5)
pd.options.display.float_format = "{:,.2f}".format

DATA_PATH = Path("../data/retail_sales_cleaned.csv")
FIG_DIR = Path("../reports/figures")
MODEL_DIR = Path("../models")
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, low_memory=False)
df["date"] = pd.to_datetime(df["date"])
print(f"Shape: {df.shape}, {df['product'].nunique()} unique products")


## Step 1 — Rank Products by Revenue and Compute Cumulative Share

In [ ]:
product_abc = df.groupby("product").agg(
    revenue=("sales_value", "sum"),
    units=("qty", "sum"),
    kg=("mass_kg", "sum"),
    n_invoices=("doc_no", "nunique"),
).reset_index()

product_abc = product_abc.sort_values("revenue", ascending=False).reset_index(drop=True)
product_abc["rank"] = product_abc.index + 1
product_abc["cum_revenue"] = product_abc["revenue"].cumsum()
product_abc["cum_pct_revenue"] = product_abc["cum_revenue"] / product_abc["revenue"].sum() * 100
product_abc["pct_of_products"] = product_abc["rank"] / len(product_abc) * 100

product_abc.head(10)


## Step 2 — Classify into A / B / C Tiers

In [ ]:
def classify_abc(cum_pct):
    if cum_pct <= 80:
        return "A"
    elif cum_pct <= 95:
        return "B"
    else:
        return "C"

product_abc["abc_class"] = product_abc["cum_pct_revenue"].apply(classify_abc)

abc_summary = product_abc.groupby("abc_class").agg(
    n_products=("product", "count"),
    total_revenue=("revenue", "sum"),
).reindex(["A", "B", "C"])
abc_summary["pct_of_products"] = abc_summary["n_products"] / abc_summary["n_products"].sum() * 100
abc_summary["pct_of_revenue"] = abc_summary["total_revenue"] / abc_summary["total_revenue"].sum() * 100

print(abc_summary)
print()
for cls, row in abc_summary.iterrows():
    print(f"Class {cls}: {row['n_products']:.0f} products ({row['pct_of_products']:.1f}% of catalogue) "
          f"generate {row['pct_of_revenue']:.1f}% of revenue.")


## Step 3 — Visualize the Pareto Curve

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

colors = {"A": "#2ca02c", "B": "#ff9900", "C": "#d62728"}
bar_colors = product_abc["abc_class"].map(colors)
ax1.bar(product_abc["rank"], product_abc["revenue"], color=bar_colors, width=1.0)
ax1.set_xlabel("Product Rank (by revenue)")
ax1.set_ylabel("Revenue per Product")
ax1.set_yscale("log")

ax2 = ax1.twinx()
ax2.plot(product_abc["rank"], product_abc["cum_pct_revenue"], color="black", linewidth=2)
ax2.axhline(80, color="gray", linestyle="--", linewidth=1)
ax2.axhline(95, color="gray", linestyle="--", linewidth=1)
ax2.set_ylabel("Cumulative % of Revenue")
ax2.set_ylim(0, 105)

handles = [plt.Rectangle((0,0),1,1, color=colors[c]) for c in ["A","B","C"]]
ax1.legend(handles, ["A products", "B products", "C products"], loc="center right")
ax1.set_title("ABC / Pareto Analysis — Product Revenue Contribution")
plt.tight_layout()
plt.savefig(FIG_DIR / "05_abc_pareto_curve.png", dpi=120)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.pie(abc_summary["n_products"], labels=[f"{c}\n({n:.0f} SKUs)" for c, n in zip(abc_summary.index, abc_summary["n_products"])],
       autopct="%1.1f%%", colors=[colors[c] for c in abc_summary.index], startangle=90)
ax.set_title("Share of Product Catalogue by ABC Class")
plt.tight_layout()
plt.savefig(FIG_DIR / "05_abc_catalogue_share.png", dpi=120)
plt.show()


## Step 4 — Class A Products (Highest Priority)

In [ ]:
class_a = product_abc[product_abc["abc_class"] == "A"].sort_values("revenue", ascending=False)
print(f"{len(class_a)} Class A products — these deserve the tightest stock control, most frequent review, "
      f"and priority supplier terms since they collectively drive 80% of revenue.")
class_a[["product", "revenue", "units", "kg", "cum_pct_revenue"]].head(20)


## Step 5 — Class C Products (Rationalization Candidates)

In [ ]:
class_c = product_abc[product_abc["abc_class"] == "C"].sort_values("revenue", ascending=False)
print(f"{len(class_c)} Class C products collectively generate only "
      f"{abc_summary.loc['C', 'pct_of_revenue']:.1f}% of revenue.")
print("These are candidates for: less frequent stock reviews, bulk/simplified ordering, "
      "or portfolio rationalization if carrying cost/shelf space is a constraint.\n")
print("Lowest-revenue Class C products (bottom 15):")
class_c.sort_values("revenue").head(15)[["product", "revenue", "units", "n_invoices"]]


## Step 6 — Cross-Tab: ABC Class vs Revenue-Volume Quadrant

Combining the ABC tier with whether a product is high/low volume adds a second, complementary inventory dimension — a Class A product that's also high-volume needs very different stock handling (frequent replenishment, careful freshness management for perishables) than a Class A product that's high-value but low-volume (premium item, less frequent but higher-stakes ordering).

In [ ]:
product_abc["kg_rank"] = product_abc["kg"].rank(pct=True)
product_abc["volume_tier"] = np.where(product_abc["kg_rank"] >= 0.5, "High Volume", "Low Volume")

cross = pd.crosstab(product_abc["abc_class"], product_abc["volume_tier"])
print(cross)

fig, ax = plt.subplots()
cross.plot(kind="bar", stacked=True, ax=ax, color=["#4c72b0", "#dd8452"])
ax.set_title("ABC Class vs Volume Tier")
ax.set_ylabel("Number of Products")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "05_abc_vs_volume.png", dpi=120)
plt.show()


## Step 7 — Save the ABC Classification

In [ ]:
output_cols = ["product", "revenue", "units", "kg", "n_invoices", "rank", "cum_pct_revenue", "abc_class", "volume_tier"]
product_abc[output_cols].to_csv(MODEL_DIR / "product_abc_classification.csv", index=False)
print(f"Saved: {MODEL_DIR / 'product_abc_classification.csv'}")


## Key Takeaways

In [ ]:
n_total = len(product_abc)
n_a = abc_summary.loc["A", "n_products"]
pct_a = abc_summary.loc["A", "pct_of_products"]
rev_a = abc_summary.loc["A", "pct_of_revenue"]
pct_c = abc_summary.loc["C", "pct_of_products"]
rev_c = abc_summary.loc["C", "pct_of_revenue"]

print(f"- {n_a:.0f} products ({pct_a:.1f}% of the {n_total}-SKU catalogue) drive {rev_a:.1f}% of total revenue.")
print(f"  This is the headline Pareto insight worth leading with in a portfolio write-up or interview.")
print(f"- Class C contains {pct_c:.1f}% of the catalogue but only {rev_c:.1f}% of revenue —")
print(f"  a strong candidate list for inventory simplification if the business wants to reduce complexity.")
print(f"- Combining ABC class with volume tier gives a sharper inventory action plan than either dimension alone:")
print(f"  high-volume Class A items need tight, frequent replenishment; low-volume Class A items are lower-frequency")
print(f"  but higher-value orders where stockouts are costly per unit.")
